# 01 — Ingestion API → MongoDB et simulation des paiements
**TP2 Big Data · BigData Commerce Analytics · B2 Sup de Vinci Rennes**

Chaîne de traitement : `API / CSV → Python → MongoDB (RAW)`

| Partie | Contenu |
|---|---|
| 0 | Connexion à MongoDB |
| 1 | Atelier 1 (1/2) : 8 ressources DummyJSON → collections `raw_*` |
| 2 | Atelier 1 (2/2) : import de `commandes_ecommerce.csv` → `raw_commandes` |
| 3 | Simulation des paiements → `raw_payments` |
| 4 | Analyse des résultats et contrôles |

**Prérequis** : MongoDB lancé (`docker start mongo`), `pip install pymongo pandas requests`.

In [ ]:
import json, random
from pathlib import Path

import requests
import pandas as pd
from pymongo import MongoClient, ASCENDING

MONGO_URI = "mongodb://localhost:27017"   # ou l'URI Atlas
DB_NAME   = "commerce"
SEED      = 42                             # graine : simulation reproductible

# Dossiers où chercher les fichiers JSON/CSV déjà téléchargés (sinon : API)
DATA_DIRS = [Path("data/raw"), Path("data/processed"), Path(".")]

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

## 0. Connexion à MongoDB

In [ ]:
client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
print("Ping :", client.admin.command("ping"))   # {'ok': 1.0} si le serveur répond
db = client[DB_NAME]
print("Base sélectionnée :", db.name)

## 1. Atelier 1 (1/2) — Charger les ressources DummyJSON

Chaque endpoint `https://dummyjson.com/<ressource>?limit=0` renvoie un objet de la forme
`{"<ressource>": [ ... ], "total": N, "skip": 0, "limit": N}` : la liste de documents se trouve sous la clé du même nom.

La fonction ci-dessous utilise le fichier JSON local s'il existe (collecte déjà faite), sinon elle interroge l'API.
On **vide la collection avant d'insérer** (`drop`) pour pouvoir relancer le notebook sans créer de doublons (idempotence).

In [ ]:
RESSOURCES = ["products", "carts", "users", "posts", "comments", "recipes", "quotes", "todos"]

def charger_ressource(nom):
    """Retourne (liste_de_documents, total_annonce, origine)."""
    for d in DATA_DIRS:
        f = d / f"{nom}.json"
        if f.exists():
            payload = json.loads(f.read_text(encoding="utf-8"))
            return payload[nom], payload.get("total"), str(f)
    r = requests.get(f"https://dummyjson.com/{nom}?limit=0", timeout=40)
    r.raise_for_status()
    payload = r.json()
    return payload[nom], payload.get("total"), "API"

bilan = []
for nom in RESSOURCES:
    docs, total_api, origine = charger_ressource(nom)
    coll = db[f"raw_{nom}"]
    coll.drop()
    coll.insert_many(docs)
    en_base = coll.count_documents({})
    bilan.append({"ressource": nom, "collection": f"raw_{nom}", "origine": origine,
                  "total_api": total_api, "recus": len(docs), "en_base": en_base,
                  "OK": len(docs) == en_base == (total_api or len(docs))})

pd.DataFrame(bilan)

> **Contrôle** : la colonne `OK` doit être `True` partout (documents reçus = documents en base = total annoncé par l'API).
>
> ⚠️ La collection `raw_users` contient des données personnelles de démonstration (adresse, e-mail, téléphone, carte bancaire…).
> C'est le sujet de réflexion du jour 3 (RGPD) : on les garde en RAW pour le TP mais on ne les exploitera pas inutilement.

## 2. Atelier 1 (2/2) — Importer le fichier des commandes

⚠️ Le fichier `commandes_ecommerce.csv` du TP doit contenir **800 lignes / 16 colonnes**.
Si tu as un fichier de seulement quelques lignes (échantillon « DEMO »), il ne convient pas : utilise la version complète
(`commandes_ecommerce_1.csv` dans tes données). La cellule choisit automatiquement le premier candidat qui a 800 lignes.

In [ ]:
CANDIDATS = [
    "data/processed/commandes_ecommerce.csv",
    "commandes_ecommerce.csv",
    "commandes_ecommerce_1.csv",
]

CSV_PATH = None
for chemin in CANDIDATS:
    if Path(chemin).exists() and len(pd.read_csv(chemin)) == 800:
        CSV_PATH = chemin
        break
assert CSV_PATH, "Aucun CSV de 800 lignes trouvé : vérifie CANDIDATS"
print("Fichier utilisé :", CSV_PATH)

cmd = pd.read_csv(CSV_PATH)
print("Forme :", cmd.shape)            # (800, 16)
cmd.head()

### Contrôles d'intégrité avant import

In [ ]:
print("Lignes                 :", len(cmd))
print("Colonnes               :", cmd.shape[1])
print("Paniers distincts      :", cmd["cart_id"].nunique())
print("Lignes dupliquées      :", cmd.duplicated().sum())
print("\nValeurs manquantes (colonnes concernées) :")
print(cmd.isna().sum()[cmd.isna().sum() > 0])

# Le total du panier est répété sur chaque ligne : il doit être unique par panier
assert cmd.groupby("cart_id")["cart_discounted_total"].nunique().max() == 1
assert cmd.groupby("cart_id")["user_id"].nunique().max() == 1

# Cohérence : somme des lignes = somme des totaux de paniers (comptés UNE fois)
somme_lignes  = round(cmd["discounted_line_total"].sum(), 2)
somme_paniers = round(cmd.groupby("cart_id")["cart_discounted_total"].first().sum(), 2)
print(f"\nSomme des lignes : {somme_lignes}  |  somme des paniers : {somme_paniers}")

> La colonne `brand` contient des valeurs manquantes : c'est normal (certains produits DummyJSON n'ont pas de marque).
> Ce n'est pas bloquant, on les stocke en `null`.

In [ ]:
# NaN pandas -> None (null MongoDB) ; on passe par JSON pour garder les types (int, float, str)
docs = json.loads(cmd.to_json(orient="records"))

db.raw_commandes.drop()
res = db.raw_commandes.insert_many(docs)
db.raw_commandes.create_index([("cart_id", ASCENDING)])

print("Insérés        :", len(res.inserted_ids))
print("En base        :", db.raw_commandes.count_documents({}))
print("Paniers en base:", len(db.raw_commandes.distinct("cart_id")))
assert db.raw_commandes.count_documents({}) == len(cmd) == 800
db.raw_commandes.find_one({}, {"_id": 0})

## 3. Simulation des paiements

### Réponses aux questions de travail

1. **Un paiement unique par panier** : le CSV contient une ligne par produit ; on regroupe donc par `cart_id`
   (`groupby`) pour obtenir 1 ligne par panier → 208 paniers = 208 tentatives. Le total du panier étant répété sur chaque ligne, on le prend **une seule fois** (`first`).
2. **Respecter 70 / 20 / 10** : on tire le statut avec `random.choices(["PAID","REFUSED","PENDING"], weights=[70,20,10])`, avec une graine fixe (`seed`) pour être reproductible.
   Sur 208 tirages, les proportions observées varient autour des valeurs théoriques : c'est normal.
3. **Éléments à reprendre** : `cart_id` (clé du paiement), `user_id` (client), `order_id` (commande), le montant du panier
   `cart_discounted_total` (montant réellement payable après remises) et la date de commande.
4. **Contrôles** : 208 paiements, un seul par `cart_id` (index unique), somme des paiements = somme des paniers, somme des 3 statuts = 208.

In [ ]:
paniers = (cmd.groupby("cart_id")
              .agg(user_id=("user_id", "first"),
                   order_id=("order_id", "first"),
                   amount=("cart_discounted_total", "first"),
                   amount_before_discount=("cart_total", "first"),
                   order_date=("order_date", "min"),      # les dates du CSV varient par ligne : on retient la plus ancienne
                   nb_lignes=("product_id", "count"))
              .reset_index()
              .sort_values("cart_id"))
print("Paniers distincts :", len(paniers))      # 208
paniers.head()

In [ ]:
# Contrôle croisé avec paniers.csv (si disponible) : mêmes montants ?
for d in DATA_DIRS:
    f = d / "paniers.csv"
    if f.exists():
        ref = pd.read_csv(f).rename(columns={"id": "cart_id"})[["cart_id", "discountedTotal"]]
        m = paniers.merge(ref, on="cart_id")
        print("Écart max avec paniers.csv :", (m["amount"] - m["discountedTotal"]).abs().max())
        break

In [ ]:
random.seed(SEED)
STATUTS = ["PAID", "REFUSED", "PENDING"]
POIDS   = [70, 20, 10]

payments = []
for p in paniers.itertuples(index=False):
    statut = random.choices(STATUTS, weights=POIDS, k=1)[0]
    montant = round(float(p.amount), 2)
    payments.append({
        "payment_id": f"PAY-{int(p.cart_id):05d}",
        "cart_id": int(p.cart_id),
        "user_id": int(p.user_id),
        "order_id": p.order_id,
        "amount": montant,                                  # montant du panier (après remises)
        "status": statut,
        "collected_amount": montant if statut == "PAID" else 0.0,   # seul PAID est encaissé
        "order_date": p.order_date,
    })

db.raw_payments.drop()
db.raw_payments.insert_many(payments)
db.raw_payments.create_index([("cart_id", ASCENDING)], unique=True)   # garantit 1 paiement / panier
print("Paiements insérés :", db.raw_payments.count_documents({}))

## 4. Analyse des résultats

In [ ]:
pay = pd.DataFrame(list(db.raw_payments.find({}, {"_id": 0})))

resume = (pay.groupby("status")
             .agg(nb_paiements=("payment_id", "count"), montant_total=("amount", "sum"))
             .reindex(STATUTS))
resume["part_%"] = (100 * resume["nb_paiements"] / resume["nb_paiements"].sum()).round(1)
resume["montant_total"] = resume["montant_total"].round(2)
resume

In [ ]:
# Même résultat avec une agrégation MongoDB native
list(db.raw_payments.aggregate([
    {"$group": {"_id": "$status", "nb": {"$sum": 1}, "montant": {"$sum": "$amount"}}},
    {"$sort": {"_id": 1}},
]))

In [ ]:
encaisse = round(pay.loc[pay["status"] == "PAID", "amount"].sum(), 2)
total_paniers = round(pay["amount"].sum(), 2)
print(f"Montant total des 208 paniers      : {total_paniers:>14,.2f}")
print(f"Montant ENCAISSÉ (PAID uniquement) : {encaisse:>14,.2f}")
print(f"Taux d'encaissement en valeur      : {100*encaisse/total_paniers:.1f} %")

### Contrôles de cohérence (aucun panier compté plusieurs fois)

In [ ]:
n = db.raw_payments.count_documents({})
assert n == 208 == paniers["cart_id"].nunique(),               "il doit y avoir 208 paiements"
assert pay["cart_id"].is_unique,                               "un panier apparaît plusieurs fois !"
assert pay["payment_id"].is_unique,                            "payment_id en double"
assert pay["status"].isin(STATUTS).all(),                      "statut inconnu"
assert resume["nb_paiements"].sum() == n,                     "la somme des statuts doit valoir 208"
assert set(pay["cart_id"]) == set(cmd["cart_id"]),             "paniers et paiements ne correspondent pas"
assert abs(pay["amount"].sum() - somme_paniers) < 0.01,        "somme des paiements != somme des paniers"
assert (pay.loc[pay["status"] != "PAID", "collected_amount"] == 0).all()

doublons = list(db.raw_payments.aggregate([
    {"$group": {"_id": "$cart_id", "n": {"$sum": 1}}}, {"$match": {"n": {"$gt": 1}}}]))
print("Paniers comptés plusieurs fois :", doublons)   # []
print("✅ Tous les contrôles sont passés")

## Conclusion du notebook 01

* 8 collections `raw_*` (ressources API), `raw_commandes` (800 lignes) et `raw_payments` (208 paiements) sont chargées dans la base `commerce`.
* Un paiement par panier, avec une répartition proche de 70 / 20 / 10.
* **Règle métier** : seuls les paiements `PAID` sont encaissés ; `REFUSED` et `PENDING` ne contribuent pas au chiffre d'affaires encaissé.

➡️ Suite : `02_traitement_pyspark.ipynb`. **Ne supprime pas la base `commerce`** (réutilisée au jour 3).